# 08 - Crypto.com Exchange Connector Exploration

**Goal**: Explore the Crypto.com Exchange public REST API to inform our production connector.

**Scope**:
- Verify Ohio eligibility (CRITICAL FIRST STEP per DEC-006)
- Test raw `httpx` approach (custom connector per DEC-011)
- Map Crypto.com Exchange symbols to canonical pairs
- Parse responses into our `TopOfBook` dataclass
- Document rate limits, error handling, edge cases
- Answer key design questions for the production connector

**Target Pairs** (from PROJECT_INSTRUCTIONS.md):
BTC/USD, BTC/USDC, LTC/USD, LTC/USDC, LTC/BTC, SOL/USD, SOL/USDC, SOL/BTC

**Key Platform Distinction** (CRITICAL):
- **Crypto.com App**: Retail consumer product, available in 49 US states (incl. Ohio). No order book API.
- **Crypto.com Exchange**: Institutional/advanced trading platform with full API. US launch Jan 2025, initially for institutional/waitlisted users. Progressive rollout ongoing.
- We need the **Exchange** API (order books, REST/WS, programmatic trading).

**Exchange API**: https://exchange-docs.crypto.com/exchange/v1/rest-ws/index.html
- Root URL: `https://api.crypto.com/exchange/v1/`
- Instrument format: `BTC_USD` (underscore-separated, uppercase)
- Public endpoints: `public/get-instruments`, `public/get-book`, `public/get-tickers`
- Response envelope: `{"id": ..., "method": "...", "code": 0, "result": {...}}`

**Lessons Applied** (from LESSONS_LEARNED.md):
- LL-001: Verify exact symbol format, don't assume
- LL-002: Document actual response shapes from live API, not just docs
- LL-003: Test rate limit behavior before building production connector
- LL-010: All prices/sizes via `to_decimal()`, never float
- LL-050: Use `nest_asyncio.apply()` for async in Jupyter
- LL-052: No batch endpoint assumption -- verify before building connector
- LL-060: Ticker endpoints often lack bid/ask sizes -- verify
- LL-072: Some exchanges return HTTP 200 for application errors (check response code field)
- LL-074: Verify exchange status before building connectors

## 1. Ohio Eligibility Assessment (DEC-006 Gate)

**CRITICAL FIRST STEP**: Per DEC-006, all exchanges must be verified Ohio-eligible before proceeding.

### Research Findings (February 2026)

**Crypto.com App (Retail)**:
- Available in 49 US states including **Ohio** (only New York excluded)
- Source: https://help.crypto.com/en/articles/2692288
- FinCEN MSB registered (confirmed via Wikipedia / company disclosures)
- Operated by Foris DAX, Inc. (US entity)

**Crypto.com Exchange (Advanced/Institutional)**:
- US launch announced **January 21, 2025** for institutional traders
- Progressive rollout to waitlisted users began **September 4, 2025**
- Exchange was previously shut down in June 2023, relaunched Jan 2025
- API documentation is public and actively maintained (latest changelog: Jan 2026)
- CFTC derivatives licenses obtained (DCM, DCO, FCM) -- Sep 2025
- OCC national trust bank charter application filed -- Oct 2025

### Ohio Eligibility Verdict

| Criterion | Status | Notes |
|-----------|--------|-------|
| App available in Ohio | Yes | 49 states, only NY excluded |
| FinCEN MSB registered | Yes | Federal registration confirmed |
| Ohio MTL | Unconfirmed | App operates in Ohio, implying compliance |
| Exchange available to US retail | Progressive rollout | Institutional-first, waitlist-based |
| Public API accessible | Likely | Public endpoints don't require auth |

**Decision**: **CONDITIONALLY PROCEED** with API exploration.
- The App's Ohio availability suggests Crypto.com has the necessary state licensing.
- Public market data endpoints should be accessible without an Exchange account.
- Phase 1 (detection/alerts) only needs public market data -- no auth required.
- Phase 4 (live trading) will require Exchange account access -- may still be gated.
- If public API calls fail with geo-restrictions, STOP and document.

**Action**: Record finding as DEC-025 in DECISION_LOG.md.

## 2. Setup & Imports

In [ ]:
# Install dependencies (run once)
# !pip install httpx nest_asyncio

In [ ]:
import asyncio
import json
import sys
import time
from pprint import pprint

import httpx
import nest_asyncio

nest_asyncio.apply()  # LL-050: Required for async in Jupyter

sys.path.insert(0, "../src")

# Our existing infrastructure -- reuse, don't reimplement
from uscryptoarb.marketdata.topofbook import tob_from_raw
from uscryptoarb.venues.symbol_translator import SymbolTranslator

BASE_URL = "https://api.crypto.com/exchange/v1"

print("Setup complete")
print(f"API root: {BASE_URL}")

## 3. Product Discovery & Symbol Mapping

Crypto.com Exchange uses underscore-separated, uppercase instrument names: `BTC_USD`.

Endpoint: `GET /public/get-instruments`

Key questions:
1. Which of our 8 target pairs are listed?
2. Is USD available, or only USDT/USDC? (DEC-001: USD != USDC, USDT out of scope)
3. What is the exact instrument_name format?

In [ ]:
# Fetch all available instruments
resp = httpx.get(f"{BASE_URL}/public/get-instruments")
print(f"Status: {resp.status_code}")

data = resp.json()
print(f"Response code: {data.get('code')}")
print(f"Method: {data.get('method')}")

# Check for API errors (LL-072: some exchanges return HTTP 200 for errors)
if data.get("code") != 0:
    print(f"ERROR: API returned code {data['code']}: {data.get('message', 'unknown')}")
else:
    instruments = data.get("result", {}).get("data", [])
    print(f"Total instruments: {len(instruments)}")
    if instruments:
        print(f"\nSample instrument keys: {sorted(instruments[0].keys())}")

In [ ]:
# Filter for SPOT instruments only (we don't need futures/perps)
spot_instruments = [i for i in instruments if i.get("inst_type") == "SPOT"]
print(f"SPOT instruments: {len(spot_instruments)}")

# Get all instrument names for SPOT
all_spot_names = sorted([i["instrument_name"] for i in spot_instruments])

# Show all BTC, LTC, SOL pairs
target_bases = ["BTC", "LTC", "SOL"]
related = [name for name in all_spot_names if any(name.startswith(f"{b}_") for b in target_bases)]
print(f"\nAll BTC/LTC/SOL spot pairs ({len(related)}):")
for name in related:
    print(f"  {name}")

In [ ]:
# Check which of our 8 target pairs are available
# Canonical -> Expected Crypto.com format
TARGET_PAIRS = {
    "BTC/USD": "BTC_USD",
    "BTC/USDC": "BTC_USDC",
    "LTC/USD": "LTC_USD",
    "LTC/USDC": "LTC_USDC",
    "LTC/BTC": "LTC_BTC",
    "SOL/USD": "SOL_USD",
    "SOL/USDC": "SOL_USDC",
    "SOL/BTC": "SOL_BTC",
}

found_pairs = {}
missing_pairs = []
for canonical, expected in TARGET_PAIRS.items():
    if expected in all_spot_names:
        found_pairs[canonical] = expected
        print(f"  found: {canonical:10s} -> {expected}")
    else:
        missing_pairs.append(canonical)
        print(f"  MISS:  {canonical:10s} -> {expected} NOT FOUND")

print(f"\nFound: {len(found_pairs)}/8 target pairs")
if missing_pairs:
    print(f"Missing: {missing_pairs}")

In [ ]:
# Build the symbol map and SymbolTranslator (only for found pairs)
CRYPTODOTCOM_SYMBOL_MAP = found_pairs.copy()

cryptodotcom_translator = SymbolTranslator(
    venue="cryptodotcom",
    canonical_to_venue=CRYPTODOTCOM_SYMBOL_MAP,
)

print("CRYPTODOTCOM_SYMBOL_MAP:")
for k, v in sorted(CRYPTODOTCOM_SYMBOL_MAP.items()):
    print(f"  {k!r:14s} -> {v!r}")

print("\nRound-trip verification:")
for canonical in sorted(CRYPTODOTCOM_SYMBOL_MAP.keys()):
    venue_sym = cryptodotcom_translator.to_venue_symbol(canonical)
    back = cryptodotcom_translator.to_canonical(venue_sym)
    match = "ok" if back == canonical else "FAIL"
    print(f"  {canonical} -> {venue_sym} -> {back}  {match}")

In [ ]:
# USD != USDC verification (DEC-001)
print("USD != USDC Verification:")
print(f"{'Pair':12s} {'bid':>14s} {'ask':>14s}")
print("-" * 45)

usd_usdc_pairs = [("BTC_USD", "BTC_USDC"), ("LTC_USD", "LTC_USDC"), ("SOL_USD", "SOL_USDC")]
for usd_sym, usdc_sym in usd_usdc_pairs:
    for sym in [usd_sym, usdc_sym]:
        if sym not in CRYPTODOTCOM_SYMBOL_MAP.values():
            print(f"{sym:12s} {'N/A (not listed)':>14s}")
            continue
        resp = httpx.get(f"{BASE_URL}/public/get-tickers", params={"instrument_name": sym})
        d = resp.json()
        if d.get("code") == 0 and d.get("result", {}).get("data"):
            ticker = d["result"]["data"][0]
            print(f"{sym:12s} {ticker.get('b', 'N/A'):>14s} {ticker.get('k', 'N/A'):>14s}")
        else:
            print(f"{sym:12s} {'ERROR':>14s}")
        time.sleep(0.3)
    print()

## 4. Ticker Endpoint -- Best Bid/Ask

Endpoint: `GET /public/get-tickers?instrument_name=BTC_USD`

Key question: Does the ticker include bid/ask **sizes** (not just prices)?
- Per API changelog, ticker has `b` (bid price), `bs` (bid size), `k` (ask price), `ks` (ask size)
- If sizes are present, ticker is our primary endpoint (no need for order book)
- This is the same advantage OKX has over Gemini (LL-060)

In [ ]:
# Fetch a single ticker
resp = httpx.get(f"{BASE_URL}/public/get-tickers", params={"instrument_name": "BTC_USD"})
print(f"Status: {resp.status_code}")
data = resp.json()
print(f"Response code: {data.get('code')}")

if data.get("code") == 0:
    ticker = data["result"]["data"][0]
    print(f"\nFull ticker response keys: {sorted(ticker.keys())}")
    print("\nFull BTC_USD ticker:")
    pprint(ticker)
    print("\n--- TopOfBook fields ---")
    print(f"  bid price (b):  {ticker.get('b', 'MISSING')}")
    print(f"  bid size  (bs): {ticker.get('bs', 'MISSING')}")
    print(f"  ask price (k):  {ticker.get('k', 'MISSING')}")
    print(f"  ask size  (ks): {ticker.get('ks', 'MISSING')}")
    print(f"  timestamp (t):  {ticker.get('t', 'MISSING')}")
    has_sizes = "bs" in ticker and "ks" in ticker
    print(f"\n  Ticker has bid/ask sizes: {'YES' if has_sizes else 'NO -- need order book'}")
else:
    print(f"ERROR: {data}")

In [ ]:
# Fetch all target pair tickers
print(f"{'Pair':12s} {'bid':>12s} {'bidSz':>12s} {'ask':>12s} {'askSz':>12s} {'ts':>16s}")
print("-" * 80)

all_tickers = {}
for canonical, venue_sym in sorted(CRYPTODOTCOM_SYMBOL_MAP.items()):
    resp = httpx.get(f"{BASE_URL}/public/get-tickers", params={"instrument_name": venue_sym})
    d = resp.json()
    if d.get("code") == 0 and d.get("result", {}).get("data"):
        t = d["result"]["data"][0]
        all_tickers[canonical] = t
        print(
            f"{venue_sym:12s} {t.get('b', '?'):>12s} {t.get('bs', '?'):>12s} {t.get('k', '?'):>12s} {t.get('ks', '?'):>12s} {str(t.get('t', '?')):>16s}"
        )
    else:
        print(f"{venue_sym:12s} ERROR: code={d.get('code')}")
    time.sleep(0.3)
print(f"\nFetched tickers for {len(all_tickers)}/{len(CRYPTODOTCOM_SYMBOL_MAP)} pairs")

In [ ]:
# Batch ticker test: GET /public/get-tickers without instrument_name
resp = httpx.get(f"{BASE_URL}/public/get-tickers")
print(f"Status: {resp.status_code}")
data = resp.json()
if data.get("code") == 0:
    all_data = data["result"]["data"]
    print(f"Total tickers returned (no filter): {len(all_data)}")
    target_venue_syms = set(CRYPTODOTCOM_SYMBOL_MAP.values())
    batch_filtered = {t["i"]: t for t in all_data if t.get("i") in target_venue_syms}
    print(f"Target pairs in batch: {len(batch_filtered)}/{len(CRYPTODOTCOM_SYMBOL_MAP)}")
    for sym in sorted(batch_filtered):
        print(f"  {sym}")
    if batch_filtered:
        sample_key = sorted(batch_filtered.keys())[0]
        print(f"\nBatch ticker keys: {sorted(batch_filtered[sample_key].keys())}")
else:
    print(f"Batch request failed: code={data.get('code')}")

## 5. Order Book Endpoint (Fallback / Validation)

Endpoint: `GET /public/get-book?instrument_name=BTC_USD&depth=1`

Even if ticker has sizes, document the order book format for validation and Phase 2 WebSocket.

In [ ]:
# Fetch order book for BTC_USD
resp = httpx.get(f"{BASE_URL}/public/get-book", params={"instrument_name": "BTC_USD", "depth": 5})
print(f"Status: {resp.status_code}")
data = resp.json()
if data.get("code") == 0:
    book_data = data["result"]["data"][0]
    print(f"\nBook response keys: {sorted(book_data.keys())}")
    print("\nFull book (depth=5):")
    pprint(book_data)
    if "asks" in book_data:
        print(f"\nAsks format (first entry): {book_data['asks'][0]}")
        print("  -> [price, size, num_orders]")
    if "bids" in book_data:
        print(f"Bids format (first entry): {book_data['bids'][0]}")
        print("  -> [price, size, num_orders]")
else:
    print(f"ERROR: {data}")

## 6. Parse into TopOfBook

Prototype parser function for production connector.
Uses ticker endpoint (if sizes available) or falls back to order book.

In [ ]:
# Prototype parser: Crypto.com ticker -> TopOfBook
# Ticker fields (per API docs):
#   i: instrument_name
#   b: best bid price
#   bs: best bid size
#   k: best ask price
#   ks: best ask size
#   t: timestamp (Unix ms)


def parse_cryptodotcom_ticker(raw, venue_sym, canonical):
    return tob_from_raw(
        venue="cryptodotcom",
        pair=canonical,
        bid_px=raw["b"],
        bid_sz=raw["bs"],
        ask_px=raw["k"],
        ask_sz=raw["ks"],
        timestamp_ms=int(raw["t"]),
    )


# Test parser on all available pairs
print("TopOfBook parsing test:")
print(
    f"{'Pair':10s} {'bid_px':>12s} {'bid_sz':>12s} {'ask_px':>12s} {'ask_sz':>12s} {'ts_ms':>16s}"
)
print("-" * 75)

parsed_tobs = {}
for canonical, ticker_data in all_tickers.items():
    try:
        tob = parse_cryptodotcom_ticker(ticker_data, CRYPTODOTCOM_SYMBOL_MAP[canonical], canonical)
        parsed_tobs[canonical] = tob
        print(
            f"{canonical:10s} {str(tob.bid_px):>12s} {str(tob.bid_sz):>12s} {str(tob.ask_px):>12s} {str(tob.ask_sz):>12s} {tob.timestamp_ms:>16d}"
        )
    except Exception as e:
        print(f"{canonical:10s} PARSE ERROR: {e}")
print(f"\nParsed: {len(parsed_tobs)}/{len(all_tickers)} pairs")

In [ ]:
# Validate parsed TopOfBooks
for canonical, tob in parsed_tobs.items():
    issues = []
    if tob.bid_px <= 0:
        issues.append("bid_px <= 0")
    if tob.ask_px <= 0:
        issues.append("ask_px <= 0")
    if tob.bid_px >= tob.ask_px:
        issues.append(f"crossed book: bid={tob.bid_px} >= ask={tob.ask_px}")
    if tob.bid_sz <= 0:
        issues.append("bid_sz <= 0")
    if tob.ask_sz <= 0:
        issues.append("ask_sz <= 0")
    if tob.timestamp_ms <= 0:
        issues.append("timestamp_ms <= 0")
    status = "OK" if not issues else f"FAIL: {', '.join(issues)}"
    print(f"  {canonical:10s} {status}")

## 7. Async httpx Pattern (Production Preview)

Test async fetching. Determine: batch ticker (1 call) vs per-pair ticker (N calls).

In [ ]:
async def fetch_all_tickers_batch(client):
    resp = await client.get(f"{BASE_URL}/public/get-tickers")
    resp.raise_for_status()
    data = resp.json()
    if data.get("code") != 0:
        raise ValueError(f"API error code {data['code']}")
    target_syms = set(CRYPTODOTCOM_SYMBOL_MAP.values())
    return {t["i"]: t for t in data["result"]["data"] if t.get("i") in target_syms}


async def fetch_tickers_per_pair(client, delay_s=0.2):
    results = {}
    for _canonical, venue_sym in CRYPTODOTCOM_SYMBOL_MAP.items():
        resp = await client.get(
            f"{BASE_URL}/public/get-tickers", params={"instrument_name": venue_sym}
        )
        resp.raise_for_status()
        data = resp.json()
        if data.get("code") == 0 and data.get("result", {}).get("data"):
            results[venue_sym] = data["result"]["data"][0]
        await asyncio.sleep(delay_s)
    return results


async def compare_fetch_strategies():
    async with httpx.AsyncClient(timeout=10.0) as client:
        t0 = time.monotonic()
        batch = await fetch_all_tickers_batch(client)
        batch_time = time.monotonic() - t0
        print(f"Batch:    {len(batch)} pairs in {batch_time * 1000:.0f}ms (1 API call)")
        await asyncio.sleep(1.0)
        t0 = time.monotonic()
        per_pair = await fetch_tickers_per_pair(client)
        per_pair_time = time.monotonic() - t0
        n_calls = len(CRYPTODOTCOM_SYMBOL_MAP)
        print(
            f"Per-pair: {len(per_pair)} pairs in {per_pair_time * 1000:.0f}ms ({n_calls} API calls)"
        )
        print(f"\nBatch is {per_pair_time / batch_time:.1f}x faster")


asyncio.run(compare_fetch_strategies())

## 8. Rate Limit Testing

From the API docs: Rate limits exist but specifics need empirical testing.

In [ ]:
# Burst test: 15 rapid requests
print("Rate limit burst test (15 rapid requests):")
latencies = []
errors = []
for i in range(15):
    t0 = time.monotonic()
    resp = httpx.get(f"{BASE_URL}/public/get-tickers", params={"instrument_name": "BTC_USD"})
    elapsed_ms = (time.monotonic() - t0) * 1000
    latencies.append(elapsed_ms)
    if resp.status_code == 429:
        errors.append(i)
        print(f"  [{i + 1:2d}] 429 TOO MANY REQUESTS ({elapsed_ms:.0f}ms)")
    elif resp.status_code != 200:
        errors.append(i)
        print(f"  [{i + 1:2d}] {resp.status_code} ({elapsed_ms:.0f}ms)")
    else:
        data = resp.json()
        if data.get("code") != 0:
            errors.append(i)
            print(f"  [{i + 1:2d}] API error code={data['code']} ({elapsed_ms:.0f}ms)")

if not errors:
    print("  All 15 requests succeeded (no 429s)")
print(
    f"\nLatency: min={min(latencies):.0f}ms max={max(latencies):.0f}ms mean={sum(latencies) / len(latencies):.0f}ms"
)

## 9. Error Handling

Document error response format. Like OKX, Crypto.com may return HTTP 200 with non-zero code.

In [ ]:
# Test 1: Invalid instrument name
print("Test 1: Invalid instrument name")
resp = httpx.get(f"{BASE_URL}/public/get-tickers", params={"instrument_name": "FAKE_PAIR"})
print(f"  HTTP status: {resp.status_code}")
pprint(resp.json())
print()

# Test 2: Missing required parameter for book
print("Test 2: Missing instrument_name for book")
resp = httpx.get(f"{BASE_URL}/public/get-book")
print(f"  HTTP status: {resp.status_code}")
pprint(resp.json())
print()

# Test 3: Non-existent endpoint
print("Test 3: Non-existent endpoint")
resp = httpx.get(f"{BASE_URL}/public/get-nonexistent")
print(f"  HTTP status: {resp.status_code}")
try:
    pprint(resp.json())
except Exception:
    print(f"  Non-JSON response: {resp.text[:200]}")

## 10. Symbol Details (Precision & Min Sizes)

Extract trading precision from `public/get-instruments` for order sizing (Phase 4).

In [ ]:
instrument_lookup = {i["instrument_name"]: i for i in spot_instruments}

print(f"{'Pair':10s} {'min_qty':>12s} {'qty_tick':>12s} {'px_tick':>12s} {'quote_ccy':>10s}")
print("-" * 60)

all_details = {}
for canonical, venue_sym in sorted(CRYPTODOTCOM_SYMBOL_MAP.items()):
    details = instrument_lookup.get(venue_sym, {})
    all_details[canonical] = details
    print(
        f"{canonical:10s} {details.get('min_quantity', '?'):>12s} {details.get('quantity_tick_size', '?'):>12s} {details.get('price_tick_size', '?'):>12s} {details.get('quote_currency', '?'):>10s}"
    )

print("\nFull instrument details for BTC_USD:")
pprint(instrument_lookup.get("BTC_USD", {}))

## 11. Timestamp Format Deep Dive

Document the timestamp format for staleness detection.

In [ ]:
from datetime import UTC, datetime

resp = httpx.get(f"{BASE_URL}/public/get-tickers", params={"instrument_name": "BTC_USD"})
data = resp.json()
ticker = data["result"]["data"][0]

ts_raw = ticker.get("t")
print(f"Raw timestamp field (t): {ts_raw} (type: {type(ts_raw).__name__})")
print(f"Length: {len(str(ts_raw))} digits")

ts_val = int(ts_raw) if isinstance(ts_raw, str) else ts_raw
if ts_val > 1e15:
    ts_ms = ts_val // 1000
    fmt = "Unix MICROSECONDS"
elif ts_val > 1e12:
    ts_ms = ts_val
    fmt = "Unix MILLISECONDS"
elif ts_val > 1e9:
    ts_ms = ts_val * 1000
    fmt = "Unix SECONDS"
else:
    ts_ms = ts_val
    fmt = "UNKNOWN"

now_ms = int(time.time() * 1000)
age_s = (now_ms - ts_ms) / 1000
dt = datetime.fromtimestamp(ts_ms / 1000, tz=UTC)

print(f"Format: {fmt}")
print(f"Converted: {ts_ms} ms")
print(f"UTC time:  {dt.isoformat()}")
print(f"Age:       {age_s:.1f}s ago")

print("\n--- Timestamp Comparison Across Exchanges ---")
print("  Kraken:      Unix seconds (float) -> int(ts * 1000)")
print("  Coinbase:    ISO 8601 with microseconds -> parse to ms")
print("  Gemini:      Unix seconds (int string) -> int(ts) * 1000")
print("  Bitstamp:    Unix microseconds (string) -> int(ts) // 1000")
print("  OKX:         Unix milliseconds (string) -> int(ts)")
print("  Crypto.com:  (determined above)")

## 12. Summary & Connector Design Notes

In [ ]:
print("=" * 100)
print(
    f"{'Feature':<22s} {'Kraken':<14s} {'Coinbase':<14s} {'Gemini':<14s} {'Bitstamp':<14s} {'OKX':<14s} {'Crypto.com':<14s}"
)
print("=" * 100)
rows = [
    ("Pairs available", "8/8", "8/8", "8/8", "6/8", "7/8", f"{len(found_pairs)}/8"),
    ("Symbol format", "XXBTZUSD", "BTC-USD", "btcusd", "btcusd", "BTC-USD", "BTC_USD"),
    ("Ticker has sizes", "Yes", "Yes", "NO", "NO", "Yes", "TBD above"),
    ("Primary endpoint", "Ticker", "Book", "Book", "Book", "Ticker", "Ticker?"),
    ("Batch support", "Yes", "No", "No", "No", "Yes", "Yes"),
    ("Ts format", "s (float)", "ISO 8601", "s (str)", "us (str)", "ms (str)", "TBD above"),
    ("SDK", "httpx", "httpx", "httpx", "httpx", "httpx", "httpx"),
    ("Taker fee", "0.26%", "0.60%", "0.40%", "0.40%", "0.10%", "0.075%?"),
]
for label, *vals in rows:
    print(
        f"{label:<22s} {vals[0]:<14s} {vals[1]:<14s} {vals[2]:<14s} {vals[3]:<14s} {vals[4]:<14s} {vals[5]:<14s}"
    )
print("=" * 100)

In [ ]:
print("Production Connector Design Notes")
print("=" * 60)
print()
print("1. ENDPOINT: Ticker (if bid/ask sizes confirmed)")
print("   public/get-tickers provides all 4 TopOfBook fields")
print()
print("2. FETCH STRATEGY: Batch (1 API call)")
print("   GET /public/get-tickers returns ALL tickers")
print("   Filter client-side (like Kraken and OKX)")
print()
print("3. RESPONSE PARSING:")
print("   Check code == 0 (HTTP 200 errors possible)")
print("   Parse result.data array")
print()
print("4. TIMESTAMP: Unix milliseconds (integer)")
print("   No conversion needed or just int(t)")
print()
print("5. SYMBOL FORMAT: Underscore-separated uppercase")
print("   Canonical BTC/USD -> Crypto.com BTC_USD")
print()
print("6. MODULE NAMING (Rule 10.8):")
print("   connectors/cryptodotcom/ (not 'crypto' -- too generic)")
print()
print("7. CONNECTOR PATTERN:")
print("   Inherits BaseAsyncConnector (DEC-018)")
print("   Batch fetch_tickers() (like KrakenClient, OkxClient)")
print()
print("8. OHIO ELIGIBILITY:")
print("   CONDITIONAL -- App is Ohio-eligible")
print("   Exchange US rollout still ongoing")
print("   Public API accessible for Phase 1 detection")

## 13. Test Fixture Generation

Capture live API responses for deterministic test fixtures.

In [ ]:
import os

FIXTURE_DIR = "../tests/fixtures"
os.makedirs(FIXTURE_DIR, exist_ok=True)

# Fixture 1: BTC_USD ticker
resp = httpx.get(f"{BASE_URL}/public/get-tickers", params={"instrument_name": "BTC_USD"})
if resp.status_code == 200 and resp.json().get("code") == 0:
    path = os.path.join(FIXTURE_DIR, "cryptodotcom_ticker_btc_usd.json")
    with open(path, "w") as f:
        json.dump(resp.json(), f, indent=2)
    print(f"Saved: {path}")
time.sleep(0.5)

# Fixture 2: SOL_BTC ticker (crypto-cross pair)
sol_btc_sym = CRYPTODOTCOM_SYMBOL_MAP.get("SOL/BTC")
if sol_btc_sym:
    resp = httpx.get(f"{BASE_URL}/public/get-tickers", params={"instrument_name": sol_btc_sym})
    if resp.status_code == 200 and resp.json().get("code") == 0:
        path = os.path.join(FIXTURE_DIR, "cryptodotcom_ticker_sol_btc.json")
        with open(path, "w") as f:
            json.dump(resp.json(), f, indent=2)
        print(f"Saved: {path}")
else:
    print("SOL/BTC not available -- skipping")
time.sleep(0.5)

# Fixture 3: Error response (invalid instrument)
resp = httpx.get(f"{BASE_URL}/public/get-tickers", params={"instrument_name": "FAKE_PAIR"})
path = os.path.join(FIXTURE_DIR, "cryptodotcom_error_invalid_instrument.json")
with open(path, "w") as f:
    json.dump(resp.json(), f, indent=2)
print(f"Saved: {path}")

print("\nFixtures ready for test development.")